### https://www.kaggle.com/competitions/drawing-with-llms

In [5]:
import kagglehub
import pandas as pd
import re


In [86]:
train_qa=pd.read_parquet('./drawing-with-llms/questions.parquet')

In [87]:
train_qa.head(8)

,id,question,choices,answer
0,02d892,What is the main setting of the image?,"[beach, desert, forest, mountain]",forest
1,02d892,Is there anything purple in the image?,"[no, yes]",yes
2,02d892,What time of day is suggested in the image?,"[dawn, dusk, midday, midnight]",dusk
3,02d892,What color is prominently featured in the image?,"[green, orange, purple, white]",purple
4,0dcd2e,What color is the coat?,"[blue, brown, gray, red]",gray
5,0dcd2e,What part of the coat has faux fur?,"[collar, hem, pockets, sleeves]",collar
6,0dcd2e,Is the coat purple?,"[no, yes]",no
7,0dcd2e,What material is the coat made of?,"[cotton, leather, silk, wool]",wool


In [88]:
import json
df_q = pd.read_parquet('./drawing-with-llms/questions.parquet')
df_q = df_q.groupby("id").agg(
    question=("question", lambda x: list(x)),  
    choices=("choices", lambda x: [list(i) for i in x]),  
    answer=("answer", lambda x: list(x))  
).reset_index()

In [89]:
df_q

,id,question,choices,answer
0,02d892,"[What is the main setting of the image?, Is th...","[[beach, desert, forest, mountain], [no, yes],...","[forest, yes, dusk, purple]"
1,0dcd2e,"[What color is the coat?, What part of the coa...","[[blue, brown, gray, red], [collar, hem, pocke...","[gray, collar, no, wool]"
2,1e9ac1,"[Is there an ocean visible in the image?, What...","[[no, yes], [inside, next to, overlooking, und...","[yes, overlooking, no, no]"
3,2b25db,"[Are the pants yellow?, Do the pants have patc...","[[no, yes], [no, yes], [no, yes], [dress, pant...","[no, yes, yes, pants]"
4,4e6a54,"[What material is the item?, Is a hat depicted...","[[corduroy, denim, leather, silk], [no, yes], ...","[corduroy, no, yes, overalls]"
5,4f1b00,[Is there any purple item present in the image...,"[[no, yes], [no, yes], [beaded, fringe, lace, ...","[yes, yes, tassel, purple]"
6,61b500,"[Is the lagoon depicted as green?, Is there a ...","[[no, yes], [no, yes], [ceiling, roof, sky, tr...","[yes, yes, sky, lagoon]"
7,65cc74,"[Is the grid's arrangement chaotic?, Are the s...","[[no, yes], [no, yes], [no, yes], [no, yes]]","[yes, no, no, yes]"
8,7c4414,"[Is the cone made of a bronze-like material?, ...","[[no, yes], [no, yes], [cone, cube, pyramid, s...","[yes, yes, cone, pyramids]"
9,996c3a,"[Is the color silver present in the image?, Wh...","[[no, yes], [opaque, reflective, solid, transl...","[yes, translucent, yes, trapezoids]"


In [90]:
# # Initialize empty dictionary to hold the final format

# multiple_choice_qa = {
#     'question': [],
#     'choices': [],
#     'answer': []
# }

# mcq=[]
# # Iterate over the DataFrame and populate the dictionary
# for i, row in df_q.iterrows():
#     multiple_choice_qa['question']=row['question']
#     multiple_choice_qa['choices']=row['choices']  # Convert the string representation of a list back to a list
#     multiple_choice_qa['answer']=row['answer']
    
#     print(multiple_choice_qa)
#     #mcq.append([multiple_choice_qa]) 
#     #appending the json into list not working peoperly, may need to convert to string and work...
    

In [2]:
import openai
import pandas as pd
import time
import os
from dotenv import load_dotenv
load_dotenv()
openai_api_key = os.getenv("OPENAI_API_KEY")
deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

In [4]:
from openai import OpenAI
client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "is it raining today in tamil nadu?"},
    ],
    stream=False
)

print(response.choices[0].message.content)

To check if it's raining today in Tamil Nadu, you can refer to real-time weather updates from reliable sources like:

1. **India Meteorological Department (IMD)** – [https://mausam.imd.gov.in](https://mausam.imd.gov.in)  
2. **AccuWeather** – [https://www.accuweather.com](https://www.accuweather.com)  
3. **Weather.com** – [https://weather.com](https://weather.com)  

Since weather conditions vary across districts (Chennai, Coimbatore, Madurai, etc.), you may need to specify a location for accurate information.  

Would you like help checking a specific city in Tamil Nadu? Let me know!


In [93]:
# Function to get summary from OpenAI API
base_model="deepseek-chat"
formatted_name = (base_model.lower().replace("/", "_").replace("-", "_").replace(".", "_"))
global base_model
def get_svg_code(dummy_text='dummy_text'):
    
    instruction  = f"""
    
        I am participating in an SVG code generation competition.
        The task involves generating SVG representations of approximately 500 short text descriptions of everyday objects and scenes across a variety of domains. Your model should generate well-formed and valid SVG code that visually represents each description.

        Here, first you need to generate a description and its correcponding svg code as given below.
        
        Details about the input descriptions:
        - Each description ranges between 10 and 200 characters, with an average of around 50 characters.
        - The content describes common, generic subjects—no brand names, trademarks, personal names, or depictions of people are included.
        - The subjects span roughly a dozen categories.
        - Half of the descriptions will focus on themes related to **landscapes**, **abstract**, and **fashion**.
        
        SVG Code Generation Requirements:
        - Your SVG output must visually and clearly reflect the provided description.
        - Only the following **SVG elements** are allowed: `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
        - Only the following **SVG attributes** are allowed: `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
        - The SVG code must be **complete, valid, and syntactically correct**. Avoid using ellipses (`...`) or placeholders.
        - Give significant importance to aesthetics part, as code is evaluated for aesthetics as well
        - Keep the visual representation **detailed yet concise, aesthetic, accurate, and compliant** with the above element and attribute constraints.
        
        Output Format:
        Please return the result in the following JSON format:
        
        <example>
        {{
            "description": "<generate description here>",
            "svg_code": "<generate full SVG code here>"
        }}
        </example>
        
        Response Here:
        """


    try:
        response = client.chat.completions.create(
            model=base_model,
            messages=[
                {"role": "system", "content": "Generate SVG code as per instruction"},
                {"role": "user", "content": instruction}
            ],
            temperature=0.7,
            max_tokens=7200
        )
        return response.choices[0].message.content.strip()
    
    except Exception as e:
        print(f"Error: {e}")
        return None

In [94]:
# # Function to get summary from OpenAI API
# from openai import OpenAI
# client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

# def get_vqa_for_topic(description):
    
#     description = description  # replace with your input
    
#     instruction = f"""
#     I am participating in an SVG code generation competition. I have trained a model to generate SVG code for a given topic. The generated code is converted into an image and evaluated using visual question answering (VQA) by an evaluation model. The questions and answers will be about the description of the generated code. 
    
#     Here is an example description and corresponding visual question answering (VQA) pair:
    
#     Example#1:
#     {{
#       'description': 'a purple forest at dusk',
#         'question': ['What is the main setting of the image?', 'Is there anything purple in the image?', 'What time of day is suggested in the image?', 'What color is prominently featured in the image?'],
#         'choices': [['beach', 'desert', 'forest', 'mountain'], ['no', 'yes'], ['dawn', 'dusk', 'midday', 'midnight'], ['green', 'orange', 'purple', 'white']],
#         'answer': ['forest', 'yes', 'dusk', 'purple']

#     }}
    
#     Example#2:
#     {{
#         'description': 'gray wool coat with a faux fur collar',
#         'question': ['What color is the coat?', 'What part of the coat has faux fur?', 'Is the coat purple?', 'What material is the coat made of?'],
#         'choices': [['blue', 'brown', 'gray', 'red'], ['collar', 'hem', 'pockets', 'sleeves'], ['no', 'yes'], ['cotton', 'leather', 'silk', 'wool']],
#         'answer': ['gray', 'collar', 'no', 'wool']
#     }}    
#     Now, generate a similar visual question answering pair for the following description:
#     '{description}'
#     """


#     try:
#         response = client.chat.completions.create(
#             model="deepseek-chat",
#             messages=[
#                 {"role": "system", "content": "You are a helpful assistant to generate visual question ansering pair"},
#                 {"role": "user", "content": instruction}
#             ],
#             temperature=0.6,
#             max_tokens=2048
#         )
#         return response.choices[0].message.content.strip()
    
#     except Exception as e:
#         print(f"Error: {e}")
#         return None

In [95]:
# from together import Together
# client = Together() # auth defaults to os.environ.get("TOGETHER_API_KEY")
# base_model="Qwen/Qwen2.5-VL-72B-Instruct"
# formatted_name = (base_model.lower().replace("/", "_").replace("-", "_").replace(".", "_"))
# global base_model

# # Function to get summary from OpenAI API
# def get_svg_code_together(dummy_text='dummy_text'):
    
#     instruction  = f"""
    
#         I am participating in an SVG code generation competition.
#         The task involves generating SVG representations of approximately 500 short text descriptions of everyday objects and scenes across a variety of domains. Your model should generate well-formed and valid SVG code that visually represents each description.

#         Here, first you need to generate a description and its correcponding svg code as given below.
        
#         Details about the input descriptions:
#         - Each description ranges between 10 and 200 characters, with an average of around 50 characters.
#         - The content describes common, generic subjects—no brand names, trademarks, personal names, or depictions of people are included.
#         - The subjects span roughly a dozen categories.
#         - Half of the descriptions will focus on themes related to **landscapes**, **abstract**, and **fashion**.
        
#         SVG Code Generation Requirements:
#         - Your SVG output must visually and clearly reflect the provided description.
#         - Only the following **SVG elements** are allowed: `svg`, `path`, `circle`, `rect`, `ellipse`, `line`, `polyline`, `polygon`, `g`, `linearGradient`, `radialGradient`, `stop`, `defs`
#         - Only the following **SVG attributes** are allowed: `viewBox`, `width`, `height`, `fill`, `stroke`, `stroke-width`, `d`, `cx`, `cy`, `r`, `x`, `y`, `rx`, `ry`, `x1`, `y1`, `x2`, `y2`, `points`, `transform`, `opacity`
#         - The SVG code must be **complete, valid, and syntactically correct**. Avoid using ellipses (`...`) or placeholders.
#         - Give significant importance to aesthetics part, as code is evaluated for aesthetics as well
#         - Keep the visual representation **concise, accurate, and compliant** with the above element and attribute constraints.
        
#         Output Format:
#         Please return the result in the following JSON format:
        
#         <example>
#         {{
#             "description": "<generate description here>",
#             "svg_code": "<generate full SVG code here>"
#         }}
#         </example>
        
#         Response Here:
#         """


#     try:
       
#         response = client.chat.completions.create(
#             model=base_model,
#             messages=[
#                 {"role": "system", "content": "Generate SVG code as per instruction"},
#                 {"role": "user", "content": instruction}
#             ],
#             temperature=0.5,
#             max_tokens=3200
#         )
#         return response.choices[0].message.content.strip()
    
#     except Exception as e:
#         print(f"Error: {e}")
#         return 0

In [96]:
from tqdm import tqdm
response_list = []
for i in tqdm(range(100), desc="Generating SVG"):
    response_list.append(get_svg_code())

Generating SVG: 100%|█████████████████████████| 100/100 [29:00<00:00, 17.41s/it]


In [97]:
df = pd.DataFrame(response_list, columns=[formatted_name])

In [98]:
import os
filename = f"{formatted_name}.csv"
counter = 1
# Keep incrementing the counter until a non-existing filename is found
while os.path.exists(filename):
    counter += 1
    filename = f"{formatted_name}_{counter}.csv"
# Now save the file
df.to_csv(filename, index=False)

In [99]:
df['deepseek_chat'].iloc[0]

'Here\'s a generated example in the requested JSON format:\n\n```json\n{\n    "description": "A simple red apple with a green leaf on a white background",\n    "svg_code": "<svg viewBox=\\"0 0 100 100\\" xmlns=\\"http://www.w3.org/2000/svg\\">\\n  <rect width=\\"100\\" height=\\"100\\" fill=\\"white\\" />\\n  <g transform=\\"translate(50, 50)\\">\\n    <path d=\\"M0,-25 A25,25 0 1,1 0,25 A25,25 0 1,1 0,-25 Z\\" fill=\\"#ff0000\\" stroke=\\"#aa0000\\" stroke-width=\\"2\\" />\\n    <path d=\\"M0,-25 Q10,-40 20,-30 L15,-15 Z\\" fill=\\"#00aa00\\" stroke=\\"#008800\\" stroke-width=\\"1\\" />\\n    <path d=\\"M0,5 Q5,15 -5,15\\" fill=\\"none\\" stroke=\\"#663300\\" stroke-width=\\"2\\" />\\n  </g>\\n</svg>"\n}\n```\n\nThis example features:\n1. A clean white background\n2. A red apple shape created with a path using arc commands\n3. A green leaf using a quadratic Bézier curve\n4. A small brown stem\n5. Simple but effective coloring with stroke outlines for definition\n6. Proper grouping and